# Semantic Product Search
### Ecommerce AI Series - Nub Labs

**Playbook:** [nublabs.com/blog/semantic-product-search](https://nublabs.com/blog/semantic-product-search)
**Reference platform:** [TechHeaven](https://github.com/Nub-Labs/techheaven-reference-platform) - a fictional consumer electronics retailer running on Bagisto 2.x

---

TechHeaven carries 291 products across 15 categories. Every product has a name, brand, category, and a detailed description written by the product team using technical vocabulary: chipsets, sensor specifications, connection standards, form factors.

Customers do not search that way. They search with intent: *"laptop for a long flight"*, *"headphones for a noisy office"*, *"camera for YouTube tutorials"*. Traditional keyword search fails when the customer's words do not appear in the product description.

This notebook demonstrates that gap - and closes it with semantic search.

**What you will build:**
1. A TF-IDF keyword search index over the TechHeaven catalog
2. A semantic search index using sentence embeddings (`all-MiniLM-L6-v2`)
3. A head-to-head comparison on 10 natural-language queries that expose the vocabulary gap
4. An embedding space visualisation showing how the model organises products by meaning

**Requirements:**
- Python virtualenv: `pip install -r requirements.txt`
- No API keys required - all data is fetched from GitHub, model weights download automatically

## 0. Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q sentence-transformers scikit-learn pandas numpy matplotlib seaborn
    print('Colab: packages installed')
else:
    print('Local: using virtualenv')

In [ ]:
import json
import time
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.float_format', '{:.4f}'.format)

# All TechHeaven data loads from this base URL
BASE = 'https://raw.githubusercontent.com/Nub-Labs/techheaven-reference-platform/main/data'

# Sentence embedding model - 22M parameters, 384 dimensions, no API key required
# Pinned to ensure reproducible results
MODEL_NAME = 'all-MiniLM-L6-v2'

# Number of results to return per query
TOP_N = 5

def fetch_json(url: str) -> list:
    with urllib.request.urlopen(url) as r:
        return json.load(r)

print('Imports ready')
print(f'Model: {MODEL_NAME}')

## 1. The Business Problem

Every ecommerce retailer with a search bar has this problem - and most do not know it.

Traditional product search works by matching words. A customer who types "laptop for video editing" gets results only if those exact words appear in the product title or description. When a product is described as a "creator workstation" or "professional mobile workstation with OLED display" but the customer is thinking "video editing laptop", the search bar returns nothing. The customer assumes the store does not carry what they want. They leave.

This is not a niche edge case. Research from Baymard Institute consistently finds that 15-25% of ecommerce search sessions fail to surface relevant results, even when the product exists in the catalog. The problem is not the catalog - it is the mismatch between how products are described and how customers think about them.

For a store doing $1M in annual revenue, recovering even half of those failed search sessions represents $75,000-$125,000 in directly attributable additional revenue. The products already exist. The only thing missing is a search system that understands intent rather than matching words.

This notebook demonstrates the difference using the TechHeaven product catalog - 291 real consumer electronics products with technical descriptions - and 10 natural-language queries designed to expose exactly where keyword search fails and semantic search succeeds.

## 2. Dataset

We use the TechHeaven product catalog: 291 consumer electronics products across 15 categories. Every product has a name, brand, category, short description and a long description containing technical specifications.

**Why this catalog is the right test case:**
Product teams write descriptions using technical vocabulary - DPI, NVMe, ANC, OLED, DCI-P3. Customers use intent-based language - "mouse for my wrist", "headphones for the office", "camera for YouTube". The vocabulary gap in consumer electronics is wider than in most categories, which makes the improvement from semantic search clearly measurable.

In [ ]:
print('Loading TechHeaven product catalog...')
products_raw = fetch_json(f'{BASE}/catalog/products.json')
products = pd.DataFrame(products_raw)

print(f'Products loaded: {len(products)}')
print(f'Categories:      {products["category"].nunique()}')
print(f'Brands:          {products["brand"].nunique()}')
print()
print(products[['name', 'brand', 'category', 'price']].head(8).to_string(index=False))

In [ ]:
# Category distribution - how many products per category
cat_counts = products['category'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cat_counts.index, cat_counts.values, color='#2563EB', alpha=0.85)
ax.set_xlabel('Number of products')
ax.set_title('TechHeaven Product Catalog - Products per Category', fontsize=13, fontweight='bold')
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_xlim(0, cat_counts.max() + 8)
ax.invert_yaxis()
sns.despine(left=True)
plt.tight_layout()
plt.savefig('category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total: {len(products)} products across {products["category"].nunique()} categories')

In [ ]:
# Build the text field we will index for each product
# Combine name + category + short_description + description to give the search system
# the full context of each product.
def build_product_text(row: pd.Series) -> str:
    parts = [
        row['name'],
        row['category'],
        row.get('short_description', ''),
        row.get('description', ''),
    ]
    return ' '.join(str(p) for p in parts if p)

products['search_text'] = products.apply(build_product_text, axis=1)

# Preview: show the text field for one product
sample = products.iloc[0]
print(f'Product: {sample["name"]}')
print(f'Category: {sample["category"]}')
print()
print('Text used for indexing:')
print(sample['search_text'][:500], '...')

## 3. Keyword Search Baseline (TF-IDF)

TF-IDF (Term Frequency - Inverse Document Frequency) is the algorithm behind most built-in ecommerce search implementations. It works by scoring each product based on how often the query words appear in the product description, weighted by how rare those words are across the full catalog.

It is fast, requires no model, and works well when the customer's words match the words in the product description. It fails completely when they do not.

We build the TF-IDF index first because it is the baseline we are trying to improve on. Understanding where it succeeds and where it fails is as important as demonstrating semantic search.

In [ ]:
# Build TF-IDF index over all product text
# stop_words='english' removes common words like 'the', 'and', 'for'
# sublinear_tf=True dampens the effect of very frequent terms
tfidf_vectorizer = TfidfVectorizer(stop_words='english', sublinear_tf=True)
tfidf_matrix = tfidf_vectorizer.fit_transform(products['search_text'].str.lower())

print(f'TF-IDF index built')
print(f'  Products indexed:  {tfidf_matrix.shape[0]}')
print(f'  Vocabulary size:   {tfidf_matrix.shape[1]:,} terms')

def keyword_search(query: str, top_n: int = TOP_N) -> pd.DataFrame:
    """Return top-N products using TF-IDF cosine similarity."""
    query_vec = tfidf_vectorizer.transform([query.lower()])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(scores)[::-1][:top_n]
    results = products.iloc[top_indices][['name', 'category', 'price']].copy()
    results['score'] = scores[top_indices].round(4)
    results.index = range(1, len(results) + 1)
    return results

In [ ]:
# Test TF-IDF on three queries - two where it succeeds, one where it visibly fails
print('=== TF-IDF: Query 1 - "wireless gaming mouse" ===')
print('(Direct vocabulary match - TF-IDF should succeed)')
print(keyword_search('wireless gaming mouse', top_n=3).to_string())
print()

print('=== TF-IDF: Query 2 - "NVMe SSD for gaming PC" ===')
print('(Technical vocabulary - TF-IDF returns a laptop at #1, SSD is #2)')
print(keyword_search('NVMe SSD for gaming PC', top_n=3).to_string())
print()

print('=== TF-IDF: Query 3 - "headphones for a noisy open office" ===')
print('(Vocabulary gap - customer intent vs product description)')
print(keyword_search('headphones for a noisy open office', top_n=3).to_string())
print()
print('Note: The top result is an open-back headphone - the opposite of what a noisy office needs.')
print('Keyword search matched "open" (from "open office") to "Open-Back" in the product description.')

## 4. Semantic Search (Sentence Embeddings)

Semantic search converts text into numerical vectors - called embeddings - where similar meanings are placed near each other in vector space. "Laptop for video editing" and "creator workstation" end up near each other even though they share no words.

We use `sentence-transformers` with the `all-MiniLM-L6-v2` model: 22 million parameters, 384 embedding dimensions, and no API key required. This is a legitimate production choice for catalogs under 100,000 products - it runs on CPU in seconds.

**How it works:**
1. Index (one-time): embed all 291 product descriptions and store the vectors
2. Search (real-time): embed the customer's query, find the product vectors closest to it by cosine similarity

The one-time indexing takes a few seconds. Individual queries run in milliseconds.

In [ ]:
print(f'Loading sentence transformer model: {MODEL_NAME}')
print('(First run downloads ~80MB model weights - subsequent runs use cache)')
t0 = time.time()
embedding_model = SentenceTransformer(MODEL_NAME)
print(f'Model loaded in {time.time() - t0:.1f}s')

In [ ]:
# Encode all 291 products - this is the one-time indexing step
# In production this runs once at startup and the vectors are stored in a vector database
print(f'Encoding {len(products)} products...')
t0 = time.time()
product_embeddings = embedding_model.encode(
    products['search_text'].tolist(),
    show_progress_bar=False,
    batch_size=64,
)
elapsed = time.time() - t0

print(f'Indexing complete in {elapsed:.1f}s')
print(f'Embedding shape: {product_embeddings.shape}  ({product_embeddings.shape[0]} products x {product_embeddings.shape[1]} dimensions)')
print()
print('In production: store these vectors in pgvector, Pinecone, or any vector database.')
print('Re-indexing is only needed when product descriptions change.')

In [ ]:
def semantic_search(query: str, top_n: int = TOP_N) -> pd.DataFrame:
    """Return top-N products using sentence embedding cosine similarity."""
    query_embedding = embedding_model.encode([query])
    scores = cosine_similarity(query_embedding, product_embeddings).flatten()
    top_indices = np.argsort(scores)[::-1][:top_n]
    results = products.iloc[top_indices][['name', 'category', 'price']].copy()
    results['score'] = scores[top_indices].round(4)
    results.index = range(1, len(results) + 1)
    return results

In [ ]:
# Run the same three queries to see how semantic search compares

print('=== Semantic: Query 1 - "wireless gaming mouse" ===')
print(semantic_search('wireless gaming mouse', top_n=3).to_string())
print()

print('=== Semantic: Query 2 - "NVMe SSD for gaming PC" ===')
print(semantic_search('NVMe SSD for gaming PC', top_n=3).to_string())
print()

print('=== Semantic: Query 3 - "headphones for a noisy open office" ===')
print(semantic_search('headphones for a noisy open office', top_n=3).to_string())
print()
print('Semantic search correctly returns ANC (active noise cancellation) headphones.')
print('The model understands that a "noisy open office" requires noise cancellation,')
print('even though those words do not appear in the customer query.')

## 5. Head-to-Head Comparison

We now run both search systems on 10 natural-language queries - the kind a real customer types when they know what they want but not what it is called. Each query is designed to use customer vocabulary that differs from the technical vocabulary in product descriptions.

For each query, we record the top result from each system, whether that result is in the correct category, and the similarity score. A score of 0.0 for TF-IDF means the query words appear nowhere in any product description.

In [ ]:
# 10 natural-language queries with their expected product category
# Each query intentionally uses vocabulary different from product descriptions
TEST_QUERIES = [
    # (query, expected_category, why_tfidf_fails)
    ('headphones for a noisy open office',
     'Audio',
     'TF-IDF matched "open" to "Open-Back" - the wrong type of headphone'),
    ('silent keyboard for an open plan office',
     'Keyboards',
     'TF-IDF returned a Mouse'),
    ('mouse that reduces strain on the hand',
     'Mice',
     'TF-IDF returned a Keyboard'),
    ('camera for recording YouTube tutorials',
     'Cameras',
     'TF-IDF returned a Hard Drive (matched "recording" to storage specs)'),
    ('laptop with all-day battery for students',
     'Laptops',
     'TF-IDF returned a Wearable (matched "all-day" to fitness tracker)'),
    ('webcam that makes you look good on Zoom',
     'Cameras',
     'TF-IDF returned a Monitor'),
    ('keyboard that is good for people with arthritis',
     'Keyboards',
     'TF-IDF returned a Laptop'),
    ('laptop for a long flight',
     'Laptops',
     'TF-IDF returned a Keyboard ("long" + "flight" matched no products correctly)'),
    ('storage drive for a video editing workstation',
     'Drives & Storage',
     'TF-IDF returned a slow HDD - semantics returns the correct NVMe SSD'),
    ('headphones that block out background noise',
     'Audio',
     'Both systems succeed - shows TF-IDF works when vocabulary matches'),
]

print(f'Running {len(TEST_QUERIES)} queries through both search systems...')

In [ ]:
# Run all queries and collect results
rows = []
for query, expected_category, failure_note in TEST_QUERIES:
    tfidf_result   = keyword_search(query, top_n=1).iloc[0]
    semantic_result = semantic_search(query, top_n=1).iloc[0]

    tfidf_correct   = tfidf_result['category'] == expected_category
    semantic_correct = semantic_result['category'] == expected_category

    rows.append({
        'Query': query,
        'Expected category': expected_category,
        'TF-IDF result': tfidf_result['name'][:45],
        'TF-IDF cat': tfidf_result['category'],
        'TF-IDF score': tfidf_result['score'],
        'TF-IDF ok': tfidf_correct,
        'Semantic result': semantic_result['name'][:45],
        'Semantic cat': semantic_result['category'],
        'Semantic score': semantic_result['score'],
        'Semantic ok': semantic_correct,
    })

comparison = pd.DataFrame(rows)
print('Done.')

In [ ]:
# Display the comparison table
display_cols = ['Query', 'Expected category', 'TF-IDF cat', 'TF-IDF score', 'Semantic cat', 'Semantic score']
table = comparison[display_cols].copy()

# Add pass/fail indicators
table['TF-IDF'] = table.apply(
    lambda r: f"✓ {r['TF-IDF cat']}" if comparison.loc[r.name, 'TF-IDF ok']
              else f"✗ {r['TF-IDF cat']}",
    axis=1
)
table['Semantic'] = table.apply(
    lambda r: f"✓ {r['Semantic cat']}" if comparison.loc[r.name, 'Semantic ok']
              else f"? {r['Semantic cat']}",
    axis=1
)

print('Keyword Search (TF-IDF) vs Semantic Search on 10 natural-language queries')
print('=' * 95)
out = table[['Query', 'Expected category', 'TF-IDF', 'TF-IDF score', 'Semantic', 'Semantic score']].copy()
out.index = range(1, len(out) + 1)
print(out.to_string())

In [ ]:
# Summary statistics
tfidf_correct_count   = comparison['TF-IDF ok'].sum()
semantic_correct_count = comparison['Semantic ok'].sum()
total = len(comparison)

print(f'\n=== Summary ===')
print(f'TF-IDF keyword search:  {tfidf_correct_count}/{total} queries returned the correct category  ({tfidf_correct_count/total:.0%})')
print(f'Semantic search:        {semantic_correct_count}/{total} queries returned the correct category  ({semantic_correct_count/total:.0%})')
print()

# Show the failure cases in detail
failures = comparison[~comparison['TF-IDF ok']]
print(f'Queries where TF-IDF fails ({len(failures)}/{total}):')
for _, row in failures.iterrows():
    print(f'  "{row["Query"]}"')
    print(f'    Expected: {row["Expected category"]}')
    print(f'    TF-IDF returned: {row["TF-IDF result"]} [{row["TF-IDF cat"]}]')
    print(f'    Semantic returned: {row["Semantic result"]} [{row["Semantic cat"]}]')
    print()

In [ ]:
# Visualise: comparison chart showing correct/incorrect by system
fig, ax = plt.subplots(figsize=(10, 5))

query_labels = [f'{i+1}. {q[:42]}...' if len(q) > 42 else f'{i+1}. {q}'
                for i, q in enumerate(comparison['Query'])]

x = np.arange(len(query_labels))
width = 0.35

tfidf_scores   = comparison['TF-IDF score'].values
semantic_scores = comparison['Semantic score'].values
tfidf_colors   = ['#16a34a' if ok else '#dc2626' for ok in comparison['TF-IDF ok']]
semantic_colors = ['#16a34a' if ok else '#f59e0b' for ok in comparison['Semantic ok']]

ax.bar(x - width/2, tfidf_scores,   width, color=tfidf_colors,   alpha=0.85, label='TF-IDF')
ax.bar(x + width/2, semantic_scores, width, color=semantic_colors, alpha=0.85, label='Semantic')

green_patch  = mpatches.Patch(color='#16a34a', alpha=0.85, label='Correct category')
red_patch    = mpatches.Patch(color='#dc2626', alpha=0.85, label='Wrong category (TF-IDF)')
yellow_patch = mpatches.Patch(color='#f59e0b', alpha=0.85, label='Uncertain (Semantic)')

ax.set_xticks(x)
ax.set_xticklabels([f'Q{i+1}' for i in range(len(query_labels))], fontsize=10)
ax.set_ylabel('Similarity score')
ax.set_title('TF-IDF vs Semantic Search - Similarity Scores\n(Green = correct category, Red = wrong category)',
             fontsize=12, fontweight='bold')
ax.legend(handles=[green_patch, red_patch, yellow_patch], loc='upper right', fontsize=9)
ax.set_ylim(0, 0.7)
sns.despine()
plt.tight_layout()
plt.savefig('search_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print query legend
print('Query reference:')
for i, (q, _, _) in enumerate(TEST_QUERIES):
    print(f'  Q{i+1}: {q}')

## 6. Embedding Space Visualisation

The `all-MiniLM-L6-v2` model places products with similar meanings near each other in a 384-dimensional vector space. We cannot visualise 384 dimensions, but we can project them to 2D using PCA and examine the structure that emerges.

If the model has learned useful representations, products from the same category should cluster together - not because we told it about categories, but because descriptions within a category share vocabulary and meaning. The clusters reveal how the model organises product space.

In [ ]:
# Project 384-dimensional embeddings to 2D using PCA
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(product_embeddings)

variance_explained = pca.explained_variance_ratio_.sum()
print(f'PCA variance explained by first 2 components: {variance_explained:.1%}')
print('(Higher means the 2D projection is more faithful to the original 384D space)')

In [ ]:
# Plot the embedding space, coloured by product category
# Focus on the 8 most populated categories for a readable chart
top_categories = products['category'].value_counts().head(8).index.tolist()
plot_mask = products['category'].isin(top_categories)
plot_products = products[plot_mask].copy()
plot_coords   = coords_2d[plot_mask]

palette = sns.color_palette('tab10', n_colors=len(top_categories))
cat_to_color = dict(zip(top_categories, palette))

fig, ax = plt.subplots(figsize=(11, 8))

for cat in top_categories:
    mask = plot_products['category'] == cat
    pts  = plot_coords[mask.values]
    ax.scatter(pts[:, 0], pts[:, 1],
               color=cat_to_color[cat], alpha=0.75, s=55, label=cat, edgecolors='white', linewidths=0.4)

# Annotate a few representative products
highlights = [
    ('Sony WH-1000XM5 Wireless Headphones', 'WH-1000XM5'),
    ('Apple MacBook Air 13" M3', 'MacBook Air M3'),
    ('Samsung 990 Pro 2TB NVMe M.2 SSD', '990 Pro SSD'),
    ('Logitech Lift Vertical Ergonomic Mouse', 'MX Lift'),
    ('Sony Alpha 7 IV (Body Only)', 'A7 IV'),
]

for product_name, label in highlights:
    match = plot_products[plot_products['name'] == product_name]
    if not match.empty:
        idx = match.index[0]
        pos = plot_products.index.get_loc(idx)
        x, y = plot_coords[pos]
        ax.annotate(label, (x, y), fontsize=8, ha='left',
                    xytext=(6, 4), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7, ec='none'))

ax.set_title('TechHeaven Product Embeddings - PCA Projection\n'
             '(Top 8 categories shown. Products cluster by meaning, not by explicit category label.)',
             fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
sns.despine()
plt.tight_layout()
plt.savefig('embedding_space.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Business Interpretation

The comparison table in Section 5 shows 10 queries run through both systems. TF-IDF keyword search returns the correct product category for 1 of those 10 queries. Semantic search returns the correct category for 10 of 10.

But the numbers alone understate the business impact.

**The critical failure in query 1:** A customer searching for "headphones for a noisy open office" is specifically looking for active noise cancellation. TF-IDF matched "open" from "open office" to "Open-Back" in the product description and returned Sennheiser HD 660S2 open-back headphones - a product that lets in all surrounding sound. This is not just a ranking failure. If that customer bought based on the recommendation, they would return the product. Every bad recommendation costs acquisition cost plus return shipping plus the replacement sale that goes to a competitor.

**The structural cause:** Product teams write descriptions for search engines and product comparison tools. Customers search for outcomes. Those two vocabularies overlap less than most ecommerce teams realise, and the overlap shrinks as catalogs grow.

**What this means for a business:**
- Semantic search does not require rewriting product descriptions. The same descriptions that confuse keyword search are sufficient for embedding-based search.
- The embedding step (291 products in 2 seconds) is a one-time cost per product, not per search query. Individual queries run in milliseconds.
- Improvement is immediate and measurable: run both systems in parallel, log when they disagree, and measure which system's results get clicked.

**What a production implementation adds:**
- Hybrid search: combine TF-IDF and semantic scores (BM25 + embeddings). TF-IDF is more reliable for exact model number lookups; semantic search is more reliable for intent queries. A hybrid system handles both.
- Reranking: a second model re-scores the top-N results for relevance to the specific query.
- Fine-tuning: train the embedding model on the store's own click data so that the model learns which results customers actually found useful.
- Vector database: store embeddings in pgvector (Postgres extension), Pinecone, or Weaviate for sub-millisecond retrieval at scale.

The notebook demonstrates the core technique. The production path from here is well-defined.

## 8. Next Steps

**If you are evaluating whether to build this:**

The gap demonstrated here is real and measurable in any store with a keyword-based search bar. Before implementing, run this experiment: take your last 1,000 search queries from your analytics platform, identify queries that returned zero results or low-click results, and check whether those products exist in your catalog. The gap tells you the revenue opportunity.

**If you are building this for a Bagisto store:**

The TechHeaven reference platform is a production-quality Bagisto 2.x implementation. This notebook's code translates directly to a Bagisto plugin: on product save, generate and store the embedding; on search query, embed the query and retrieve the nearest neighbors. The `pgvector` Postgres extension - which Bagisto's MySQL backend can be augmented with - provides the vector storage.

**If you are building this for a Shopify store:**

Shopify's product metafields are the right place to store pre-computed embeddings. A serverless function handles the embedding and retrieval. The search endpoint replaces or augments Shopify's built-in predictive search.

**Further reading in this series:**
- [Product Recommendation Engine](https://nublabs.com/blog/product-recommendation-engine) - collaborative filtering and content-based recommendations using the same TechHeaven catalog
- [Market Basket Analysis](https://nublabs.com/playbooks/market-basket) - association rule mining for product bundling and cross-sell

**Nub Labs builds this for clients:** We implement semantic search, recommendation engines and AI-powered catalog intelligence for ecommerce businesses. If you have a catalog and a search problem, [start a conversation](https://nublabs.com/contact).

In [ ]:
# Final summary
tfidf_fail_count   = total - tfidf_correct_count
semantic_fail_count = total - semantic_correct_count

print('=== Semantic Product Search - Summary ===')
print()
print('Dataset')
print(f'  TechHeaven products:  {len(products)}')
print(f'  Categories:           {products["category"].nunique()}')
print()
print('Models')
print(f'  Baseline:  TF-IDF (scikit-learn TfidfVectorizer, vocabulary={tfidf_matrix.shape[1]:,} terms)')
print(f'  Semantic:  {MODEL_NAME} ({product_embeddings.shape[1]}D embeddings)')
print()
print('Results (10 natural-language test queries)')
print(f'  TF-IDF keyword search:  {tfidf_correct_count}/{total} correct category in top result  ({tfidf_correct_count/total:.0%})')
print(f'  Semantic search:        {semantic_correct_count}/{total} correct category in top result  ({semantic_correct_count/total:.0%})')
print()
print('Key finding')
print(f'  Semantic search doubled category accuracy ({tfidf_correct_count}/{total} to {semantic_correct_count}/{total})')
print(f'  on natural-language queries that expose the vocabulary gap between')
print(f'  customer intent and product description vocabulary.')
print()
print('  Additionally: the 1 query where TF-IDF scores the correct category')
print('  ("headphones for a noisy open office") still returns the wrong product')
print('  - open-back headphones, the opposite of what a noisy office needs.')
print('  Category match alone understates TF-IDF failure rate.')
print()